## Problem statement
Implement the forward pass (and, as a bonus, full BPTT) of a simple LSTM with a toy input of two time steps and compare with `torch.nn.LSTM`.

$f_t=\sigma(W_f[h_{t-1},x_t]+b_f)$, $i_t=\sigma(W_i[h_{t-1},x_t]+b_i)$, $\tilde C_t=\tanh(W_C[h_{t-1},x_t]+b_C)$
$C_t=f_t* C_{t-1}+i_t*\tilde C_t$, $o_t=\sigma(W_o[h_{t-1},x_t]+b_o)$, $h_t=o_t*\tanh(C_t)$

Output for classification: $\hat y=\sigma(W_y h_2+b_y)$, binary cross-entropy $L=-[y\log\hat y+(1-y)\log(1-\hat y)]$. Input size = hidden size = 1.

In [1]:
import numpy as np
np.set_printoptions(precision=8, suppress=True)
sigmoid = lambda z: 1 / (1 + np.exp(-z))

## S1: Input sequence and parameters

In [2]:
rng = np.random.default_rng(1)
nx, nh = 1, 1
mkW = lambda: rng.normal(0, 0.5, (nh, nh + nx))      # acts on the concatenation [h_{t-1}, x_t]
mkb = lambda: rng.normal(0, 0.1, (nh, 1))

Wf, Wi, Wc, Wo = mkW(), mkW(), mkW(), mkW()
bf, bi, bc, bo = mkb(), mkb(), mkb(), mkb()
Wy, by = rng.normal(0, 0.5, (1, nh)), np.zeros((1, 1))   # output layer

P = [Wf, Wi, Wc, Wo, bf, bi, bc, bo, Wy, by]
pnames = ["Wf", "Wi", "Wc", "Wo", "bf", "bi", "bc", "bo", "Wy", "by"]
P_init = [p.copy() for p in P]

xs = [np.array([[0.5]]), np.array([[-0.8]])]          # x1, x2
y = 1.0                                               # binary target
h0, C0 = np.zeros((nh, 1)), np.zeros((nh, 1))

## S2: Forward pass

In [3]:
def lstm_forward(xs, P, verbose=False):
    Wf, Wi, Wc, Wo, bf, bi, bc, bo, Wy, by = P
    h, C, cache = h0, C0, []
    for t, x in enumerate(xs, 1):
        v  = np.vstack([h, x])                         # [h_{t-1}, x_t]
        f  = sigmoid(Wf @ v + bf)
        i  = sigmoid(Wi @ v + bi)
        g  = np.tanh(Wc @ v + bc)                      # C~_t
        Cn = f * C + i * g
        o  = sigmoid(Wo @ v + bo)
        hn = o * np.tanh(Cn)
        cache.append(dict(v=v, f=f, i=i, g=g, o=o, C_prev=C, C=Cn, h=hn))
        if verbose:
            print(f"t={t}: f={f.ravel()} i={i.ravel()} C~={g.ravel()} o={o.ravel()}")
            print(f"     C{t}={Cn.ravel()}  h{t}={hn.ravel()}")
        h, C = hn, Cn
    z = Wy @ h + by
    yhat = sigmoid(z)
    L = -(y * np.log(yhat) + (1 - y) * np.log(1 - yhat)).item()
    return L, yhat, cache

L0, yhat, cache = lstm_forward(xs, P, verbose=True)
print("\ny_hat =", yhat.ravel(), "  BCE loss =", L0)

t=1: f=[0.5601724] i=[0.42644362] C~=[0.11393894] o=[0.54982183]
     C1=[0.04858853]  h1=[0.02669403]
t=2: f=[0.42859704] i=[0.63531489] C~=[-0.16217906] o=[0.45389567]
     C2=[-0.08220987]  h2=[-0.03723087]

y_hat = [0.5034273]   BCE loss = 0.6863159679728189


## Bonus: backward pass (BPTT) and gradient check
With $z=W_yh_T+b_y$: $\partial L/\partial z=\hat y-y$. For each step (from $T$ down to 1), with $v=[h_{t-1};x_t]$:

* $\delta o=\delta h*\tanh(C_t)$, $\;\delta C \mathrel{+}= \delta h*o*(1-\tanh^2C_t)$
* $\delta f=\delta C*C_{t-1}$, $\delta i=\delta C*\tilde C$, $\delta\tilde C=\delta C*i$, $\;\delta C_{t-1}=\delta C*f$
* pre-activation grads: $\delta z_f=\delta f*f(1-f)$, $\delta z_i=\delta i*i(1-i)$, $\delta z_C=\delta\tilde C*(1-\tilde C^2)$, $\delta z_o=\delta o*o(1-o)$
* $\delta W_\bullet \mathrel{+}= \delta z_\bullet v^T$, $\delta b_\bullet \mathrel{+}= \delta z_\bullet$, $\;\delta v=\sum_\bullet W_\bullet^T\delta z_\bullet$, and the first $n_h$ entries of $\delta v$ are $\delta h_{t-1}$.

In [4]:
def lstm_backward(P, cache, yhat):
    Wf, Wi, Wc, Wo, bf, bi, bc, bo, Wy, by = P
    G = [np.zeros_like(p) for p in P]
    dz = yhat - y
    G[8] = dz @ cache[-1]["h"].T; G[9] = dz.copy()
    dh = Wy.T @ dz
    dC = np.zeros((nh, 1))
    for c in reversed(cache):
        do = dh * np.tanh(c["C"])
        dC = dC + dh * c["o"] * (1 - np.tanh(c["C"]) ** 2)
        df, di, dg = dC * c["C_prev"], dC * c["g"], dC * c["i"]
        dzf, dzi = df * c["f"] * (1 - c["f"]), di * c["i"] * (1 - c["i"])
        dzc, dzo = dg * (1 - c["g"] ** 2), do * c["o"] * (1 - c["o"])
        for k, dzk in zip([0, 1, 2, 3], [dzf, dzi, dzc, dzo]):
            G[k] += dzk @ c["v"].T; G[4 + k] += dzk
        dv = Wf.T @ dzf + Wi.T @ dzi + Wc.T @ dzc + Wo.T @ dzo
        dh = dv[:nh]
        dC = dC * c["f"]
    return G

grads = lstm_backward(P, cache, yhat)

def num_grad(k, eps=1e-6):
    g = np.zeros_like(P[k])
    for idx in np.ndindex(*P[k].shape):
        Pp = [p.copy() for p in P]; Pm = [p.copy() for p in P]
        Pp[k][idx] += eps; Pm[k][idx] -= eps
        g[idx] = (lstm_forward(xs, Pp)[0] - lstm_forward(xs, Pm)[0]) / (2 * eps)
    return g

for k, n in enumerate(pnames):
    print(f"{n}: analytic={grads[k].ravel()}  numeric={num_grad(k).ravel()}  match={np.allclose(grads[k], num_grad(k), atol=1e-6)}")

Wf: analytic=[ 0.00002619 -0.00078476]  numeric=[ 0.00002619 -0.00078476]  match=True
Wi: analytic=[-0.00008269  0.00315186]  numeric=[-0.00008269  0.00315186]  match=True
Wc: analytic=[ 0.00136129 -0.03062038]  numeric=[ 0.00136129 -0.03062038]  match=True
Wo: analytic=[-0.00009924  0.00311683]  numeric=[-0.00009924  0.00311683]  match=True
bf: analytic=[0.00098095]  numeric=[0.00098095]  match=True
bi: analytic=[-0.00175003]  numeric=[-0.00175003]  match=True
bc: analytic=[0.07134881]  numeric=[0.07134881]  match=True
bo: analytic=[-0.00343243]  numeric=[-0.00343243]  match=True
Wy: analytic=[0.01848783]  numeric=[0.01848783]  match=True
by: analytic=[-0.4965727]  numeric=[-0.4965727]  match=True


## S3: Same network in PyTorch
PyTorch stacks the gates in the order **(i, f, g, o)** and splits each weight into `weight_ih` (acts on $x_t$) and `weight_hh` (acts on $h_{t-1}$). Our $W_\bullet=[W_{hh}\;|\;W_{ih}]$ (columns for $h$ first, then $x$), so we slice accordingly. `bias_hh` is set to 0 (and frozen) so the effective bias equals ours.

In [5]:
import torch, torch.nn as nn
dt = torch.float64
T = lambda a: torch.tensor(np.asarray(a), dtype=dt)

order = [Wi, Wf, Wc, Wo]                               # PyTorch gate order: i, f, g, o
border = [bi, bf, bc, bo]
lstm = nn.LSTM(input_size=1, hidden_size=1, num_layers=1, dtype=dt)
fc = nn.Linear(1, 1, dtype=dt)
with torch.no_grad():
    lstm.weight_ih_l0.copy_(T(np.vstack([W[:, nh:] for W in order])))
    lstm.weight_hh_l0.copy_(T(np.vstack([W[:, :nh] for W in order])))
    lstm.bias_ih_l0.copy_(T(np.vstack(border).ravel()))
    lstm.bias_hh_l0.zero_()
    fc.weight.copy_(T(Wy)); fc.bias.copy_(T(by.ravel()))
lstm.bias_hh_l0.requires_grad_(False)

X = T([xs[0].item(), xs[1].item()]).view(2, 1, 1)      # (T, batch, feat)
h_init = torch.zeros(1, 1, nh, dtype=dt); c_init = torch.zeros(1, 1, nh, dtype=dt)

out, (hn, cn) = lstm(X, (h_init, c_init))
_, (h1_t, c1_t) = lstm(X[:1], (h_init, c_init))        # state after step 1 only (to print C1)
yhat_t = torch.sigmoid(fc(hn[-1]))
loss_t = nn.BCELoss()(yhat_t, torch.tensor([[y]], dtype=dt))
loss_t.backward()

print("PyTorch  h1 =", h1_t.item(), " C1 =", c1_t.item())
print("PyTorch  h2 =", hn.item(),   " C2 =", cn.item())
print("PyTorch  y_hat =", yhat_t.item(), " BCE =", loss_t.item())

PyTorch  h1 = 0.026694032244113697  C1 = 0.048588532728751105
PyTorch  h2 = -0.037230866231132  C2 = -0.08220986752041719
PyTorch  y_hat = 0.5034272992704888  BCE = 0.6863159679728189


## S4: Comparison and discussion

In [6]:
print(f"{'Quantity':10s}{'NumPy':>20s}{'PyTorch':>20s}   match")
rows = [("h1", cache[0]["h"].item(), h1_t.item()), ("C1", cache[0]["C"].item(), c1_t.item()),
        ("h2", cache[1]["h"].item(), hn.item()),   ("C2", cache[1]["C"].item(), cn.item()),
        ("y_hat", yhat.item(), yhat_t.item()),     ("BCE loss", L0, loss_t.item())]
for n, a, b in rows: print(f"{n:10s}{a:20.12f}{b:20.12f}   {np.isclose(a, b, atol=1e-10)}")

# gradient comparison (map our gate order f,i,c,o -> PyTorch rows i,f,g,o)
gate_row = {"f": 1, "i": 0, "c": 2, "o": 3}
gi, gh, gb = lstm.weight_ih_l0.grad.numpy(), lstm.weight_hh_l0.grad.numpy(), lstm.bias_ih_l0.grad.numpy()
print("\nGradients:")
for k, name in enumerate(["f", "i", "c", "o"]):
    r = slice(gate_row[name] * nh, (gate_row[name] + 1) * nh)
    gW = np.hstack([gh[r], gi[r]]); gB = gb[r].reshape(-1, 1)
    print(f"dW{name}: match={np.allclose(grads[k], gW, atol=1e-10)}   db{name}: match={np.allclose(grads[4+k], gB, atol=1e-10)}")
print("dWy: match=", np.allclose(grads[8], fc.weight.grad.numpy()), "  dby: match=", np.allclose(grads[9], fc.bias.grad.numpy().reshape(1, 1)))

Quantity                 NumPy             PyTorch   match
h1              0.026694032244      0.026694032244   True
C1              0.048588532729      0.048588532729   True
h2             -0.037230866231     -0.037230866231   True
C2             -0.082209867520     -0.082209867520   True
y_hat           0.503427299270      0.503427299270   True
BCE loss        0.686315967973      0.686315967973   True

Gradients:
dWf: match=True   dbf: match=True
dWi: match=True   dbi: match=True
dWc: match=True   dbc: match=True
dWo: match=True   dbo: match=True
dWy: match= True   dby: match= True


### Discussion
The hidden states, cell states, output probability, BCE loss and (bonus) all gradients of the from-scratch LSTM agree with `torch.nn.LSTM` to round-off, so PyTorch's LSTM follows exactly the equations above (forget gate, input gate, candidate cell, cell update, output gate, hidden state). The only differences are bookkeeping ones: PyTorch orders the gates as (i, f, g, o), splits each weight matrix into input-to-hidden and hidden-to-hidden parts, and keeps two bias vectors per gate whose sum is the effective bias.

The additive cell update $C_t=f_t*C_{t-1}+i_t*\tilde C_t$ gives gradients a path (through $f_t$) that avoids repeated squashing by tanh, which is why LSTMs mitigate the vanishing-gradient problem of simple RNNs.